#Understanding the Transformer

1. What problem does the paper solve?
   
Imagine asking an AI to complete this sentence:

The animal didn't cross the street because it was too tired.

To understand the sentence, the model must connect “it” with “the animal.”

Traditional sequence models, such as RNNs and LSTMs, process text sequentially. 

**Transformers use attention mechanisms to help each word identify other relevant words, even when they are far apart.**

For example:
Input sentence

The animal didn't cross the street because it was tired


Illustrative example: attention can learn a strong relationship between “it” and “animal” based on context.


2. The main differences: RNN vs Transformer
Feature	RNN / LSTM	Transformer
Processing	Sequential	Can process input positions in parallel during training
Long-distance relationships	Can be difficult to learn	Self-attention directly connects distant tokens
Main mechanism	Recurrent state	Attention
Training parallelism	Limited across sequence positions	Much greater across sequence positions
Modern LLMs	Not the standard architecture	Core architecture for many LLMs

 **Self-attention mechanism:**
Think of it like a database lookup: you type in a Query (what you are looking for), match it against Keys (the index/titles of data), and retrieve the most relevant Values (the actual content).
Here is how those three steps break down:

1. Calculating Relevance Scores (The Matching Step)

The mechanism first determines how much attention every word should pay to every other word in the sentence.
    * The Math: It takes the Query vector (\(Q\)) of a specific word and computes the dot product with the Key vector (\(K\)) of every word in the sequence.
    * The Formula Portion: \(Q \cdot K^T\)
    * The Result: Raw, unscaled similarity scores. A high score means the two words are highly relevant to each other contextually.


    
2. Converting to Weights Using Softmax (The Standardizing Step)

The raw relevance scores can be any positive or negative number. To turn them into usable probabilities, they must be normalized.
• The Math: The raw scores are divided by a scaling factor (the square root of the key vector dimension, \(\sqrt{d_{k}}\)) to prevent gradients from exploding. Then, the Softmax function is applied.
• The Formula Portion: \(\text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)\)
• The Result: All scores are converted into a probability distribution between 0 and 1, and they all add up to exactly 1 (or 100%). These are your final attention weights.



In [1]:
pip install numpy

Note: you may need to restart the kernel to use updated packages.


In [2]:
import numpy as np

# Three tokens, each represented by a vector of 2 numbers
X = np.array([
    [1.0, 0.0],  # Token 1: cat
    [0.0, 1.0],  # Token 2: milk
    [1.0, 1.0]   # Token 3: thirsty
])

# Simplified, manually chosen weight matrices
W_Q = np.array([
    [1.0, 0.0],
    [0.0, 1.0]
])

W_K = np.array([
    [1.0, 0.0],
    [0.0, 1.0]
])

W_V = np.array([
    [1.0, 0.0],
    [0.0, 1.0]
])

# Step 1: Create Queries, Keys and Values
Q = X @ W_Q
K = X @ W_K
V = X @ W_V

# Step 2: Calculate attention scores
d_k = K.shape[1]
scores = (Q @ K.T) / np.sqrt(d_k)

# Step 3: Calculate softmax safely
shifted_scores = scores - np.max(
    scores, axis=1, keepdims=True
)

exp_scores = np.exp(shifted_scores)

attention_weights = exp_scores / np.sum(
    exp_scores, axis=1, keepdims=True
)

# Step 4: Calculate context-aware output vectors
output = attention_weights @ V

print("Queries (Q):")
print(np.round(Q, 3))

print("\nKeys (K):")
print(np.round(K, 3))

print("\nValues (V):")
print(np.round(V, 3))

print("\nAttention scores:")
print(np.round(scores, 3))

print("\nAttention weights:")
print(np.round(attention_weights, 3))

print("\nAttention output:")
print(np.round(output, 3))

Queries (Q):
[[1. 0.]
 [0. 1.]
 [1. 1.]]

Keys (K):
[[1. 0.]
 [0. 1.]
 [1. 1.]]

Values (V):
[[1. 0.]
 [0. 1.]
 [1. 1.]]

Attention scores:
[[0.707 0.    0.707]
 [0.    0.707 0.707]
 [0.707 0.707 1.414]]

Attention weights:
[[0.401 0.198 0.401]
 [0.198 0.401 0.401]
 [0.248 0.248 0.503]]

Attention output:
[[0.802 0.599]
 [0.599 0.802]
 [0.752 0.752]]


In [3]:
import numpy as np

weights = np.array([0.75, 0.25])

V = np.array([
    [2.0, 4.0],
    [6.0, 8.0]
])

result = weights @ V

print(result)

[3. 5.]



AI Mode conversation: self attention is it a model


You said: self attention is it a model

self attention is it a model
No, self-attention is not a standalone model; it is a neural network building block or mathematical mechanism.

Key Differences

• What it is: Self-attention is a layer or function that computes relationships between different elements (like words in a sentence) within a single sequence. It transforms input tokens into context-aware representations by calculating Query (Q), Key (K), and Value (V) vectors.
• Where it is used: It is the core engine inside larger model architectures—most notably the Transformer architecture introduced in the 2017 paper "Attention Is All You Need".
• Models vs. Mechanisms: Complete machine learning models (such as BERT, GPT-4, or T5) are built using self-attention layers combined with feed-forward networks, embeddings, and normalization layers.
Would you like a quick breakdown of how Query, Key, and Value work inside a self-attention layer?
• 
IBM
What is self-attention? | IBM
Definition: Self-attention is a machine learning attention mechanism weighing token importance to evaluate relationships within in...
• 
GeeksforGeeks
Self - Attention in NLP - GeeksforGeeks
Self - Attention in NLP * Self-attention helps a model understand how words in a sentence are related to each other. It allows the...
• 
Sebastian Raschka, PhD
Coding Self
Self-Attention The concept of “attention” in deep learning has its roots in the effort to improve Recurrent Neural Networks (RNNs)
Show all

first input sentence is converted into tokens right

Yes, exactly. Before a model or a self-attention layer can process a sentence, the raw text must be converted into tokens.
Here is the exact step-by-step pipeline of how a sentence goes from words to data the self-attention layer can understand:

1. Tokenisation (The Split)

The raw string of text is broken down into smaller pieces called tokens.
• Tokens are not always whole words; they are often sub-words or characters.
• Example: The sentence "Self-attention is great" might be split into ["Self", "-", "attention", "is", "great"].


2. Numerical Indexing (The ID)

Computers don't understand letters, so each unique token is mapped to a specific number (an ID) from the model's predefined vocabulary dictionary.
• ["Self", "-", "attention", "is", "great"] → [1045, 999, 4321, 2003, 2307]

3. Embedding (The Vector)

These IDs are then looked up in a massive table to convert each token into a dense vector (a long list of numbers, like 512 or 768 values). This gives the tokens their initial semantic meaning.

4. Positional Encoding (The Order)

Because self-attention processes all tokens at the exact same time, it has no inherent sense of word order. A mathematical signature representing the position of each word is added to the embedding vectors.
Only after these four steps are complete are the vectors finally fed into the self-attention layer to calculate the Queries, Keys, and Values.

        7. What you should remember from this lesson
        - Self-attention lets each token gather information from other tokens in the sequence.
        - Q, K and V are learned transformations of the token representations.
        - Dot products calculate Query–Key similarity scores.
        - Softmax converts scores into attention weights.
        - Weighted sums of Values create context-aware output vectors.
        One important distinction: self-attention calculates relationships between tokens; it does not automatically guarantee that the model understands language correctly. The model learns useful representations through training.

                The formula from the original paper
                The paper defines multi-head attention as follows. 
                
                NeurIPS Proceedings
                +1
                
                
                
                \[
                \operatorname{MultiHead}(Q,K,V)
                =\operatorname{Concat}(head_1,\ldots,head_h)W^O
                \]
                
                Each head performs its own attention calculation:
                \[
                head_i=\operatorname{Attention}
                (QW_i^Q,KW_i^K,VW_i^V)
                \]
                
                Let's translate this into plain English.
                Symbol	Meaning


                Let's translate this into plain English.
                Symbol	Meaning
                \(h\)	Number of attention heads
                \(W_i^Q\)	Query projection matrix for head \(i\)
                \(W_i^K\)	Key projection matrix for head \(i\)
                \(W_i^V\)	Value projection matrix for head \(i\)
                Concat	Join the head outputs along their feature dimensions
                \(W^O\)	Learned output projection matrix
                
                There are three major stages:
                1. Project the inputs into separate Q, K and V vectors for each head.
                2. Calculate attention independently in every head.
                3. Concatenate the outputs and apply a final learned projection.
                In the original paper's base Transformer, there are 8 attention heads, with 64-dimensional Keys and Queries and 64-dimensional Values per head.

In [4]:
## Step 1: Create input data and projection matrices
import numpy as np

# Three tokens, each with four features
X = np.array([
    [1.0, 0.0, 1.0, 0.0],
    [0.0, 1.0, 0.0, 1.0],
    [1.0, 1.0, 0.0, 0.0]
])

# Two heads; each head has 2 features per Q, K and V
rng = np.random.default_rng(42)

Wq1 = rng.normal(size=(4, 2))
Wk1 = rng.normal(size=(4, 2))
Wv1 = rng.normal(size=(4, 2))

Wq2 = rng.normal(size=(4, 2))
Wk2 = rng.normal(size=(4, 2))
Wv2 = rng.normal(size=(4, 2))

# Final output projection
Wo = rng.normal(size=(4, 4))

In [5]:
#Step 2: Create a function for one attention head

def scaled_dot_product_attention(Q, K, V):
    d_k = K.shape[1]

    # Calculate Query-Key scores
    scores = Q @ K.T / np.sqrt(d_k)

    # Softmax, calculated row by row
    scores = scores - scores.max(axis=1, keepdims=True)
    exp_scores = np.exp(scores)
    weights = exp_scores / exp_scores.sum(
        axis=1, keepdims=True
    )

    # Weighted sum of Value vectors
    output = weights @ V

    return output, weights

In [6]:
# Head 1: independent Q, K, V projections
Q1 = X @ Wq1
K1 = X @ Wk1
V1 = X @ Wv1

head1, weights1 = scaled_dot_product_attention(
    Q1, K1, V1
)

# Head 2: different Q, K, V projections
Q2 = X @ Wq2
K2 = X @ Wk2
V2 = X @ Wv2

head2, weights2 = scaled_dot_product_attention(
    Q2, K2, V2
)

# Concatenate the two heads
concatenated = np.concatenate(
    [head1, head2], axis=1
)

# Apply the final output projection
output = concatenated @ Wo

print("Head 1 attention weights:")
print(np.round(weights1, 3))

print("\nHead 2 attention weights:")
print(np.round(weights2, 3))

print("\nConcatenated shape:", concatenated.shape)
print("Final output shape:", output.shape)
print("\nFinal output:")
print(np.round(output, 3))

head1 = np.array([[1, 2],
                  [3, 4]])

head2 = np.array([[5, 6],
                  [7, 8]])

concatenated1 = np.concatenate(
    [head1, head2], axis=1
)

print("\nConcatenated shape:", concatenated1)

Head 1 attention weights:
[[0.478 0.19  0.331]
 [0.231 0.442 0.328]
 [0.179 0.484 0.337]]

Head 2 attention weights:
[[0.302 0.364 0.334]
 [0.148 0.295 0.556]
 [0.318 0.339 0.342]]

Concatenated shape: (3, 4)
Final output shape: (3, 4)

Final output:
[[1.097 0.751 0.599 2.041]
 [0.895 0.467 0.685 2.003]
 [0.848 0.665 0.557 2.143]]

Concatenated shape: [[1 2 5 6]
 [3 4 7 8]]


**Chapter 3: Positional Encoding**

Our goal is to understand why Transformers need positional encoding, how the original paper implements it, and how to calculate it using NumPy.

1. Why do we need positional encoding?

Consider these two sentences:
- Sentence A: The dog chased the cat.
- Sentence B: The cat chased the dog.
- 
Both sentences contain the same words, but their meanings are different because the word order changes.

Remember from our previous lessons: self-attention calculates relationships between tokens. However, standard self-attention alone does not inherently know which token comes first, second, or third.

Positional encoding provides information about each token's position in the sequence.


2. How does positional encoding work?

   **The original Transformer paper uses sinusoidal positional encoding, based on sine and cosine functions.**
            
            
            They produce smooth, repeating wave patterns.
            
            The paper assigns each token position a vector. Different dimensions of the vector use different frequencies, allowing the model to distinguish positions.
            
            The formula is:
                       $$ PE(pos,2i)=\sin\left(\frac{pos}{10000^{2i/d_{\text{model}}}}\right) $$

            
            \[
            PE(pos,2i+1)=\cos\left(\frac{pos}{10000^{2i/d_{\text{model}}}}\right)
            \]

               Let's decode the symbols.
            Symbol	                Meaning
            pos	                    Position of the token: 0, 1, 2, 3, ...
            i	                    Index used to select a pair of dimensions
            \(d_{\text{model}}\)	Embedding dimension
            sin	                    Used for even dimensions
            cos	                    Used for odd dimensions

You don't need to memorize the formula yet. The key idea is that every position gets a different numerical pattern.

In [7]:
import numpy as np

def positional_encoding(max_length, d_model):
    positions = np.arange(max_length)[:, np.newaxis]

    dimensions = np.arange(d_model)[np.newaxis, :]

    angles = positions / np.power(
        10000,
        (2 * (dimensions // 2)) / d_model
    )

    encoding = np.zeros((max_length, d_model))

    encoding[:, 0::2] = np.sin(angles[:, 0::2])
    encoding[:, 1::2] = np.cos(angles[:, 1::2])

    return encoding


pe = positional_encoding(max_length=4, d_model=4)

print(np.round(pe, 3))
print("Shape:", pe.shape)

[[ 0.     1.     0.     1.   ]
 [ 0.841  0.54   0.01   1.   ]
 [ 0.909 -0.416  0.02   1.   ]
 [ 0.141 -0.99   0.03   1.   ]]
Shape: (4, 4)



**Feed-Forward Neural Networks**
    
Let's continue our five-chapter sprint!
    
In Chapter 3, we learned how positional encoding adds information about word order.
    
Now we'll study another important component of the Transformer: the Position-wise Feed-Forward Network (FFN).
    
1. What happens after self-attention?
    
    Remember the main Transformer flow:
   
    1. Token embeddings are created.
    2. Positional encodings are added.
    3. Self-attention combines information from relevant tokens.
       
    4. **The feed-forward network transforms the resulting representations.**
    
    Self-attention helps each token gather information from other tokens. The feed-forward network then processes each token's representation using a small neural network.
    
    Think of it this way:
    - Self-attention: Gather relevant information.
    - Feed-forward network: Process and transform that information.

            The mathematical formula
            The original paper defines the feed-forward network as:
            \[
            \operatorname{FFN}(x)=
            \max(0,xW_1+b_1)W_2+b_2
            \]
            
            Here is what each component means:
            Component	    Meaning
            \(x\)	        Input token representation
            \(W_1, b_1\)	First layer's weights and bias
            \(\max(0,z)\)	ReLU activation
            \(W_2, b_2\)	Second layer's weights and bias
            Output	Transformed token representation
            
            The ReLU activation function is simple:
            \[
            \operatorname{ReLU}(x)=\max(0,x)
            \]
            
            Examples:
            - ReLU(5) = 5
            - ReLU(0) = 0
            - ReLU(-3) = 0
            Why use ReLU? Without a non-linear activation between the two linear layers, their composition would still be a single linear transformation. The activation lets the network learn more complex transformations.

In [8]:
import numpy as np

values = np.array([-3, -1, 0, 2, 5])

result = np.maximum(0, values)

print(result)

[0 0 0 2 5]


In [9]:
import numpy as np

# Three tokens, four features per token
X = np.array([
    [0.2, 0.4, 0.1, 0.7],
    [0.5, 0.3, 0.8, 0.2],
    [0.9, 0.1, 0.4, 0.6]
])

# Reproducible random weights
rng = np.random.default_rng(42)

# First layer: 4 features -> 8 features
W1 = rng.normal(0, 0.2, size=(4, 8))
b1 = np.zeros(8)

# Second layer: 8 features -> 4 features
W2 = rng.normal(0, 0.2, size=(8, 4))
b2 = np.zeros(4)

# First linear transformation
hidden = X @ W1 + b1

# Apply ReLU
activated = np.maximum(0, hidden)

# Second linear transformation
output = activated @ W2 + b2

print("Input shape:", X.shape)
print("Hidden shape:", hidden.shape)
print("Output shape:", output.shape)

print("\nOutput:")
print(np.round(output, 3))

Input shape: (3, 4)
Hidden shape: (3, 8)
Output shape: (3, 4)

Output:
[[-0.013  0.132 -0.071  0.076]
 [ 0.021  0.123 -0.06   0.099]
 [ 0.023  0.151 -0.062  0.106]]


Important concept: Position-wise processing
        Why is it called a position-wise feed-forward network?
        
        Because the same feed-forward network is applied independently to every token position.
        
        For example, if the attention output contains 10 tokens, the network processes all 10 token vectors using the same  
            $$ 
            (W_1\), \(W_2\), \(b_1\), and \(b_2\). 
            $$
        
        Unlike self-attention, the feed-forward network does not directly mix information between different token positions. It transforms the features at each position.


        Chapter 4 recap
        You now understand the feed-forward component of a Transformer:
        - Two linear layers transform the token features.
        - ReLU introduces non-linearity.
        - The hidden dimension expands and then returns to the model dimension.
        - The same network is applied independently at each token position.

**Chapter 5: Residual Connections and Layer Normalization**
    
Great! Let's continue with the Attention Is All You Need study plan. Today we've completed positional encoding and feed-forward networks. Now we'll learn two important techniques that help Transformers train effectively:
    
1. Residual connections — help information and gradients flow through deep networks.
2. Layer normalization — helps keep the values inside the network well-scaled.
    
These concepts are used throughout Transformer encoder and decoder layers.
    
1. Why do we need these techniques?
    
Imagine a Transformer with many layers. Each layer processes the output from the previous layer.
    
Without careful design, two problems can occur:
- Important information from earlier layers may get lost.
- Training a deep network can become difficult because gradients may become too small or too large.
    
Residual connections and layer normalization help address these problems.

2. Residual connections:
   A residual connection adds a layer's input to the output produced by that layer.

                                    The formula is:
                                         $$y = x + F(x)$$
            
            Here, \(x\) is the input and \(F(x)\) is the result of processing that input through a sublayer.
            
Example: Suppose the input is 10 and the sublayer produces 3.
- Without a residual connection: output = 3
- With a residual connection: output = \(10+3=13\)
  
The original input contributes directly to the output instead of relying entirely on the sublayer.

3. Layer normalization
Layer normalization normalizes values across the features of each token's representation.

For a vector \(x\), the simplified formula is:
            $$
            \operatorname{LayerNorm}(x)=
            \frac{x-\mu}{\sqrt{\sigma^2+\epsilon}}\gamma+\beta
            \$$

Where:
- \\mu\ = mean  of  the  vector's  features.
- \(\sigma^2\) = variance of the vector's features.
- \(\epsilon\) = a small number that prevents division by zero.
- \(\gamma\) and \(\beta\) = learnable scale and shift parameters.
For now, we'll set \(\gamma=1\) and \(\beta=0\) to understand the basic operation.

That means:
1. The input goes through the sublayer.
2. The sublayer output is added to the original input.
3. Layer normalization is applied to the sum.


Layer normalization calculates statistics **separately for each token,** across its feature values. It doesn't calculate one mean across the entire batch of tokens.


Chapter 5 recap
- Residual connection: adds the original input to the sublayer output.
- Layer normalization: normalizes feature values within each token representation.
- Together: help Transformers preserve information and train deep networks more effectively.

**Chapter 6: The Transformer Encoder**

1. What does a Transformer encoder do?
A Transformer encoder reads a sequence of tokens and creates contextual representations of them.

For example, consider this sentence:
"The animal didn't cross the street because it was tired."

To understand the word it, the model needs to relate it to other words in the sentence. 
Self-attention helps the encoder build those relationships.

The encoder transforms the input representations into richer representations that capture context.


2. Architecture of one encoder layer
            Input embeddings + positional encoding
            
            Multi-head self-attention
            
            Add residual connection + layer normalization
            Position-wise feed-forward network
            
            Add residual connection + layer normalization
            Encoder output


A Transformer encoder layer has two main sublayers:
1. Multi-head self-attention.
2. Feed-forward neural network.
Each sublayer has a residual connection and layer normalization in the original paper's design.

A complete Transformer stacks multiple encoder layers. The original paper used six.


3. Understand the data flow
Suppose our input contains three tokens, and each token has four features.

Our input matrix has shape:
                    $$
                    X \in \mathbb{R}^{3\times4}
                    $$

- 3 rows = 3 tokens.
- 4 columns = 4 features per token.
The steps are:

Step 1 — Add positional information. The encoder needs to know token positions because self-attention alone doesn't inherently encode word order.

Step 2 — Apply multi-head self-attention. Each token can gather information from other tokens in the sequence.

Step 3 — Add the residual and normalize. Combine the attention output with its input, then normalize.

Step 4 — Apply the feed-forward network. Process each token's representation independently through the same neural network.

Step 5 — Add another residual and normalize. This produces the output of the encoder layer.